# Diagonal integration: [multiple RNA, multiple ATAC]

Diagonal integration combines RNA and ATAC measured in different cells, with no pairing between them. This tutorial is for data in several batches: three RNA batches and three ATAC batches. It runs iNMF and online_iNMF on `D37mini`, 3,000 RNA and 3,000 ATAC profiles of the benchmark dataset `D37`, then on data in your own format.

iNMF, online_iNMF and scJoint read ATAC as gene-activity scores, made beforehand with a tool such as Signac or ArchR. GLUE reads the peak matrix. The dataset holds both.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_multi.ipynb).

## 1. Install

This cell installs `multibench-sc`. It keeps the numpy and pandas that are already installed, so Colab needs no restart.

<details>
<summary>Details</summary>

On Colab, choose a GPU runtime before you run the notebook: Runtime -> Change runtime type -> T4 GPU. On a CPU runtime, an environment that has a smaller CPU build gets that build, and training methods run slower.

</details>

In [ ]:
import numpy, pandas
%pip install -q multibench-sc numpy=={numpy.__version__} pandas=={pandas.__version__}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D37mini` (12 MB) once. Each method runs in its own environment. `mtb.env.install` downloads the environments for iNMF and online_iNMF, with no conda needed. The download is 0.9 GB.

<details>
<summary>Details</summary>

`state` is `PACKED` for an environment downloaded now and `have` for one that was already there. Without `dry_run=False`, `mtb.env.install` downloads nothing and returns the plan with its sizes.

Environments go to `mtb.config.DEFAULT.envs_dir`. To use another disk, set it before this cell.

From a terminal, `multibench env install --methods iNMF,online_iNMF --packed --run` does the same.

</details>

In [ ]:
import pandas as pd
import multibench as mtb

mtb.data.fetch("D37mini")

In [ ]:
METHODS = ["iNMF", "online_iNMF"]
envs = mtb.env.install(METHODS, dry_run=False)
pd.DataFrame(envs)[["env", "methods", "state"]]

## 3. Run the methods

`run_all` runs each method on `D37mini` and scores its output with the scIB metrics. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

Each method writes an embedding: a table of numbers with one row per cell. `run_all` scores it against the cell-type labels of `D37mini`.

The clustering metrics `ARI`, `NMI`, `ASW`, `iASW`, `iF1` and `cLISI` measure how well the embedding separates the cell types. The batch metrics `ASW_batch`, `GC` and `iLISI` measure how well the batches mix. They appear only when the data has several batches.

`res.failures` says why a method failed. `params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_batch("out/D37mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run_all("D37mini", "diagonal", methods=METHODS,
                  out_dir="out/D37mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size shows the rank within a column, and bigger is better. The fill compares a value with the other rows in the same column.

<details>
<summary>Details</summary>

The lightest fill is the lowest value in this figure, not zero.

Metrics are grouped by family: blue for dimension reduction and clustering, green for batch correction. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and the note under the figure names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA counts and ATAC as one AnnData per batch, with a cell type for each cell. `mtb.io.export_dataset` with `batch_index=` writes one RNA batch and one ATAC batch per call. iNMF and online_iNMF read ATAC as gene-activity scores, and online_iNMF needs about 5,000 cells in total.

<details>
<summary>Details: export</summary>

The RNA batches and the ATAC batches are numbered separately. RNA batch 1 and ATAC batch 1 need not come from the same sample.

For ATAC as a peak matrix, pass `atac_kind="peak"`. GLUE reads the peak matrix.

The folder name, `MYDIAG`, is the dataset name for `run_all`, and `data_path` is the folder that holds it.

`mtb.scan("MYDIAG", "diagonal", data_path="mydata")` checks the folder and the environments without running anything. Its `reason` column says what is missing.

`overwrite=True` replaces the files of an earlier run of this cell. Without it, `export_dataset` raises `FileExistsError` rather than replace a file.

</details>

Here the first 900 RNA profiles and the first 900 ATAC profiles of each `D37mini` batch take the place of your data:

In [ ]:
d = mtb.config.DEFAULT.data_path / "D37mini"
n = 900
rna, atac = [], []
for b in (1, 2, 3):
    r = mtb.io.read_canonical(d / f"rna{b}.h5")
    r.obs["celltype"] = pd.read_csv(d / f"rna_cty{b}.csv")["x"].values
    a = mtb.io.read_canonical(d / f"atac_gas{b}.h5")
    a.obs["celltype"] = pd.read_csv(d / f"atac_cty{b}.csv")["x"].values
    rna.append(r[:n].copy())
    atac.append(a[:n].copy())
rna, atac

Write the folder, then run the same methods on it:

In [ ]:
for b in (1, 2, 3):
    mtb.io.export_dataset(rna[b - 1], "mydata/MYDIAG", atac=atac[b - 1],
                          atac_kind="gene_activity", labels="obs:celltype",
                          category="diagonal", batch_index=b, overwrite=True)

In [ ]:
mine = mtb.run_all("MYDIAG", "diagonal", methods=METHODS, data_path="mydata",
                   out_dir="out/MYDIAG")
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

4 methods have a variant for diagonal [multiple RNA, multiple ATAC]. Each ran on `D37mini` in the package's own check, on a server with a GPU:

| Method | Result | Seconds |
|---|---|---|
| GLUE | scored | 567 |
| iNMF | scored | 16 |
| online_iNMF | scored | 8 |
| scJoint | scored | 43 |

To run them all, set `METHODS` to these names in section 2 and run the notebook again. That downloads 3 envs, 4.8 GB to download on a CPU host, 8.4 GB on a GPU host. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_multi_all.ipynb) does the same in Colab.

The article also evaluates Conos on this task. Its published script reads a third input file in another format, so the package does not run it on more than two files.

## Troubleshooting

`res.failures` lists each method that failed or was skipped. Its `error` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| `env.install` refuses on macOS or Windows | methods run only on Linux: use Colab or a Linux machine |
| a method needs an NVIDIA GPU | choose a GPU runtime on Colab, or a machine with a GPU |
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run_all` |
| low `label_order_confidence` | several label files fit the cell count: check `label_order_candidates` in `res.results` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- `mtb.cite(METHODS)` returns the citations for the benchmark and the methods you ran.
- The other tutorials: [vertical RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt/), [vertical RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_atac/), [vertical RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt_atac/), [diagonal [RNA, ATAC]](https://dsichang.github.io/scMultiBench/tutorials/diagonal_rna_atac/), [mosaic [RNA, RNA + ADT, ADT]](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_adt/), [mosaic [RNA, RNA + ATAC, ATAC]](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_atac/), [mosaic Mixed, with shared modality](https://dsichang.github.io/scMultiBench/tutorials/mosaic_shared/), [mosaic Mixed, without shared modality](https://dsichang.github.io/scMultiBench/tutorials/mosaic_unshared/), [cross Multiple RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_adt/), [cross Multiple RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_atac/), [cross Multiple ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/cross_adt_atac/), [cross Multiple RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_adt_atac/).
- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.